### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
# Dependency bootstrap (auto-install missing packages)
import importlib.util
import subprocess
import sys

PKG_MAP = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}

missing = [pkg for mod, pkg in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing missing dependencies: {missing}')
    if sys.platform == 'emscripten':
        import micropip
        await micropip.install(missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready.')


# Interactive k-NN Example 1

This notebook mirrors the **Example 1** plot in the lecture and demonstrates how k-NN classification changes when a new point is moved.

For the website, an interactive exported view is embedded at:

- `assets/notebooks/knn_example1_interactive.html`


In [ ]:
import math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

points = [
    ("A", 1.0, 2.0, 0),
    ("B", 2.0, 3.5, 0),
    ("E", 1.5, 4.0, 0),
    ("C", 4.0, 1.0, 1),
    ("D", 4.0, 4.0, 1),
    ("F", 3.5, 2.0, 1),
]

def classify(new_x, new_y, k):
    d = []
    for name, x, y, cls in points:
        dist = math.hypot(new_x - x, new_y - y)
        d.append((dist, name, x, y, cls))
    d.sort(key=lambda t: t[0])
    neighbors = d[:k]
    c0 = sum(1 for n in neighbors if n[-1] == 0)
    c1 = sum(1 for n in neighbors if n[-1] == 1)
    pred = 1 if c1 > c0 else 0
    if c0 == c1:
        pred = neighbors[0][-1]
    return pred, neighbors

def plot_knn(new_x=3.0, new_y=2.5, k=3):
    pred, neighbors = classify(new_x, new_y, int(k))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 5)
    ax.grid(True, linestyle='--', alpha=0.4)

    for name, x, y, cls in points:
        color = 'blue' if cls == 0 else 'red'
        ax.scatter([x], [y], c=color, marker='x', s=250)
        ax.text(x + 0.08, y + 0.08, name, fontsize=12)

    ax.scatter([new_x], [new_y], c='purple', marker='x', s=280)
    ax.text(new_x + 0.08, new_y + 0.08, 'New', fontsize=12)

    for dist, name, x, y, cls in neighbors:
        ax.plot([new_x, x], [new_y, y], color='gray', linewidth=1)

    pred_label = 'Class 0' if pred == 0 else 'Class 1'
    ax.set_title(f'k-NN Example 1 | k={k} | Predicted: {pred_label}')
    plt.show()

widgets.interact(
    plot_knn,
    new_x=widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=3.0),
    new_y=widgets.FloatSlider(min=0.0, max=5.0, step=0.1, value=2.5),
    k=widgets.IntSlider(min=1, max=len(points), step=1, value=3),
)
